In [ ]:
!git clone https://github.com/MarioAlessandroNapoli/neuro-llm.git
%cd neuro-llm
!pip install -q -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

In [ ]:
from huggingface_hub import snapshot_download, whoami

hf_user = whoami()["name"]
snapshot_download(f"{hf_user}/tinystories-tokenized", repo_type="dataset", local_dir="data")

In [ ]:
# Benchmark throughput (gruppo bench), 4 config x 20M token. Riferimento storico: ~110k tok/s.
# A: base (fused AdamW ora sempre attivo su CUDA) . B: +torch.compile
# C: DDP 2 GPU batch globale invariato (16x2 = 32, ricetta salva) . D: DDP 2 GPU batch 64 (32x2)
for name, flags in [
    ('bench-a-1gpu-b32',      '--devices 1 --batch-size 32'),
    ('bench-b-1gpu-b32-comp', '--devices 1 --batch-size 32 --compile'),
    ('bench-c-2gpu-b16-comp', '--devices 2 --batch-size 16 --compile'),
    ('bench-d-2gpu-b32-comp', '--devices 2 --batch-size 32 --compile'),
]:
    !python -m src.train --arch transformer --tokens 20000000 --seed 1 --lr 1e-3 --group bench --run-name {name} {flags}
